[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/fastapi-certified/notebooks/day-03-pydantic-validation.ipynb#scrollTo=aa11bb22)

---
# Day 3 · Pydantic Models — Validation and Serialization
**certified-journeys / fastapi-certified** · Day 3 · Data Validation

> **Goal for today:** Build production-quality Pydantic v2 models with field-level constraints, nested structures, custom validators, and strict-mode configuration — and integrate them with FastAPI routes.


In [ ]:
%pip install -q fastapi uvicorn[standard] httpx pydantic


## Step 1 · `BaseModel` — Required vs Optional Fields

Every Pydantic model inherits from `BaseModel`. Field optionality is controlled by **whether you provide a default**:

| Declaration | Behaviour |
|---|---|
| `name: str` | Required — must be present in input |
| `name: str = "default"` | Optional — falls back to default if absent |
| `name: str \| None = None` | Optional and nullable — can be `null` in JSON |
| `name: Optional[str] = None` | Same as above (Python 3.9 syntax) |

Pydantic v2 also supports `Field(...)` for richer metadata:

```python
from pydantic import Field
price: float = Field(..., ge=0, description="Price in USD, must be non-negative")
```

`Field(...)` — the `...` (Ellipsis) means the field is **required** but adds extra metadata.


In [ ]:
from pydantic import BaseModel, Field
from typing import Optional
import json

class Product(BaseModel):
    # Required fields — no default provided
    sku: str                                   # stock-keeping unit, always required
    name: str                                  # product name, always required
    price: float                               # price in USD, always required

    # Optional fields — have defaults
    description: Optional[str] = None         # nullable, defaults to None
    quantity: int = 0                          # stock count, defaults to 0
    is_active: bool = True                     # listing status, defaults to True

# Valid instantiation — required fields only, defaults fill the rest
p1 = Product(sku="SKU-001", name="Notebook", price=12.99)
print("Required only:", p1.model_dump())

# Full instantiation
p2 = Product(
    sku="SKU-002",
    name="Pen",
    price=1.50,
    description="Ballpoint pen, blue ink",
    quantity=500,
    is_active=True,
)
print("Full instance:", p2.model_dump())

# Round-trip: dict → model → dict
raw = {"sku": "SKU-003", "name": "Eraser", "price": "0.99"}  # price as string
p3 = Product.model_validate(raw)  # Pydantic coerces "0.99" → 0.99 (float)
print("Coerced price type:", type(p3.price).__name__, "→ value:", p3.price)


### What just happened?

- **`model_validate(dict)`** is the Pydantic v2 name for `Model(**dict)` with additional input-mode control. It accepts dicts, ORM objects (with `from_attributes=True`), and JSON strings.
- **Type coercion** is on by default: `"0.99"` → `0.99`, `"1"` → `True`, `1` → `True`. This is convenient but can hide bugs. Use `model_config = ConfigDict(strict=True)` to disable coercion (Step 5).
- **`model_dump()`** replaces Pydantic v1's `.dict()`. It returns a Python dict. Use `model_dump_json()` for a JSON string.
- Fields **without defaults** are required at construction time — Pydantic raises `ValidationError` if they are missing.


## Step 2 · Field-Level Constraints — `Field()` Annotations

Pydantic v2 supports rich field constraints via `Field()`:

| Constraint | Type | Effect |
|---|---|---|
| `min_length=N` | `str` | String must be at least N characters |
| `max_length=N` | `str` | String must be at most N characters |
| `pattern=r"..."` | `str` | String must match regex |
| `ge=N` | numeric | Value ≥ N (greater-than-or-equal) |
| `gt=N` | numeric | Value > N (strictly greater-than) |
| `le=N` | numeric | Value ≤ N |
| `lt=N` | numeric | Value < N |
| `multiple_of=N` | numeric | Value must be a multiple of N |
| `min_length=N` | `list` | List must have at least N items |
| `max_length=N` | `list` | List must have at most N items |


In [ ]:
from pydantic import BaseModel, Field, ValidationError

class StrictProduct(BaseModel):
    sku: str = Field(
        ...,
        min_length=3,
        max_length=20,
        pattern=r"^SKU-[0-9]{3,6}$",   # must match SKU-NNN to SKU-NNNNNN
        description="Stock-keeping unit in format SKU-NNN",
    )
    name: str = Field(..., min_length=1, max_length=100)
    price: float = Field(..., gt=0, le=100_000, description="Price must be > 0 and ≤ 100,000")
    quantity: int = Field(default=0, ge=0, description="Cannot be negative")
    tags: list[str] = Field(default=[], max_length=10, description="At most 10 tags")

# Valid model
good = StrictProduct(sku="SKU-001", name="Widget", price=9.99)
print("Valid product:", good.model_dump())

# Test each constraint type
test_cases = [
    ({"sku": "X", "name": "Test", "price": 1.0},        "sku too short"),
    ({"sku": "SKU-001", "name": "Test", "price": -5.0},  "price negative"),
    ({"sku": "ABC-001", "name": "Test", "price": 1.0},   "sku wrong pattern"),
    ({"sku": "SKU-001", "name": "Test", "price": 1.0, "quantity": -1}, "quantity negative"),
]

for data, label in test_cases:
    try:
        StrictProduct.model_validate(data)
        print(f"  {label}: PASSED (unexpected)")
    except ValidationError as e:
        # Extract the first error's message for brevity
        err = e.errors()[0]
        print(f"  {label}: REJECTED → [{err['loc']}] {err['msg']}")


### What just happened?

- **`Field(...)` constraints are evaluated before your code runs** — Pydantic rejects invalid data immediately with a `ValidationError` containing all failing fields (not just the first one).
- **`pattern=r"..."` (regex)** replaces Pydantic v1's `regex=` argument. The regex is compiled once at model definition time.
- **`gt=0` vs `ge=0`:** use `gt` when zero is invalid (e.g., price must be positive), `ge` when zero is allowed (e.g., quantity can be zero).
- **`e.errors()`** returns a list of dicts — each with `loc`, `msg`, `type`, and `input`. In FastAPI, this list becomes the `detail` array in the 422 response body.


## Step 3 · Nested Models — Composition Over Flat Structures

Pydantic models can **contain other Pydantic models** as field types. This maps cleanly to nested JSON objects and is how you model real-world domain concepts:

```json
{
  "order_id": "ORD-001",
  "customer": {"name": "Alice", "email": "alice@example.com"},
  "line_items": [
    {"sku": "SKU-001", "quantity": 2, "unit_price": 9.99},
    {"sku": "SKU-002", "quantity": 1, "unit_price": 24.99}
  ]
}
```

Pydantic validates the **entire nested structure** recursively. A missing field anywhere in the tree surfaces as a validation error with the full path in `loc`.


In [ ]:
from pydantic import BaseModel, Field, EmailStr
from pydantic import field_validator
from typing import Optional
import json

# Leaf-level model — used as a field in other models
class Customer(BaseModel):
    name: str = Field(..., min_length=1)
    email: str = Field(..., pattern=r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$")

# Another leaf-level model
class LineItem(BaseModel):
    sku: str = Field(..., min_length=3)
    quantity: int = Field(..., ge=1, description="Must order at least 1")
    unit_price: float = Field(..., gt=0)

    @property
    def subtotal(self) -> float:
        """Computed property — not stored, calculated on access."""
        return round(self.quantity * self.unit_price, 2)

# Top-level model — nests Customer and a list of LineItem
class Order(BaseModel):
    order_id: str = Field(..., pattern=r"^ORD-[0-9]+$")
    customer: Customer                              # nested single model
    line_items: list[LineItem] = Field(..., min_length=1)  # nested list, at least 1 item
    notes: Optional[str] = None

    @property
    def total(self) -> float:
        """Aggregate computed property across nested list."""
        return round(sum(item.subtotal for item in self.line_items), 2)

# Build an order from a raw dict (simulating a JSON request body)
raw_order = {
    "order_id": "ORD-001",
    "customer": {"name": "Alice", "email": "alice@example.com"},
    "line_items": [
        {"sku": "SKU-001", "quantity": 2, "unit_price": 9.99},
        {"sku": "SKU-002", "quantity": 1, "unit_price": 24.99},
    ],
}

order = Order.model_validate(raw_order)
print("Order ID:     ", order.order_id)
print("Customer:     ", order.customer.name, "<" + order.customer.email + ">")
print("Line items:   ", len(order.line_items))
for item in order.line_items:
    print(f"  {item.sku:10} qty={item.quantity}  unit=${item.unit_price}  subtotal=${item.subtotal}")
print("Order total:  $", order.total)

# Serialize the whole nested structure to dict
print("\nmodel_dump():")
print(json.dumps(order.model_dump(), indent=2))


### What just happened?

- **Nested models are validated recursively** — Pydantic validates `Customer` and each `LineItem` independently, then validates the `Order` that contains them.
- **`list[LineItem]`** with `min_length=1` ensures the order has at least one item. An empty `line_items: []` raises a validation error.
- **`@property`** methods on Pydantic models work normally — they are not included in `model_dump()` by default (they are not fields), which keeps the serialized form clean.
- **`model_dump()` on a nested model** recursively converts all nested models to dicts — perfect for JSON serialization or database persistence.


## Step 4 · Custom Validators — `@field_validator`

When built-in constraints (`min_length`, `ge`, `pattern`) are not enough, use `@field_validator` to write arbitrary Python validation logic.

Pydantic v2 `@field_validator` signature:

```python
@field_validator("field_name", mode="before" | "after")
@classmethod
def validate_field(cls, v):
    # v is the raw value (mode="before") or parsed value (mode="after")
    if not condition:
        raise ValueError("Descriptive error message")
    return v  # or return a transformed value
```

| Mode | When it runs | `v` type |
|---|---|---|
| `"before"` | Before Pydantic's type parsing | raw input (often `str`) |
| `"after"` | After Pydantic's type parsing | parsed type (`int`, `str`, etc.) |


In [ ]:
from pydantic import BaseModel, Field, field_validator, ValidationError
import re

class UserRegistration(BaseModel):
    username: str = Field(..., min_length=3, max_length=30)
    email: str
    password: str = Field(..., min_length=8)
    age: int = Field(..., ge=13, le=120)

    @field_validator("username", mode="after")
    @classmethod
    def username_lowercase_only(cls, v: str) -> str:
        """Usernames must be lowercase alphanumeric with underscores only."""
        if not re.match(r"^[a-z0-9_]+$", v):
            raise ValueError(
                "Username must contain only lowercase letters, digits, and underscores"
            )
        return v  # return the (optionally transformed) value

    @field_validator("email", mode="after")
    @classmethod
    def email_must_be_lowercase(cls, v: str) -> str:
        """Normalize email to lowercase and validate basic format."""
        v = v.lower().strip()  # transform: normalize before validation
        if "@" not in v or "." not in v.split("@")[-1]:
            raise ValueError("Invalid email format")
        return v

    @field_validator("password", mode="after")
    @classmethod
    def password_complexity(cls, v: str) -> str:
        """Require at least one digit and one uppercase letter."""
        if not any(c.isdigit() for c in v):
            raise ValueError("Password must contain at least one digit")
        if not any(c.isupper() for c in v):
            raise ValueError("Password must contain at least one uppercase letter")
        return v

# Valid registration
user = UserRegistration(
    username="alice_99",
    email="  ALICE@EXAMPLE.COM  ",  # gets normalized to lowercase
    password="Secure123",
    age=28,
)
print("Valid user:", user.model_dump())
print("Email normalized:", user.email)  # lowercased + stripped

# Test validator rejections
bad_cases = [
    ({"username": "Alice-99", "email": "a@b.com", "password": "Secure123", "age": 25}, "uppercase in username"),
    ({"username": "alice",    "email": "a@b.com", "password": "nodigits",  "age": 25}, "no digit in password"),
    ({"username": "alice",    "email": "a@b.com", "password": "nouppercase1", "age": 25}, "no uppercase in password"),
]

for data, label in bad_cases:
    try:
        UserRegistration.model_validate(data)
    except ValidationError as e:
        err = e.errors()[0]
        print(f"  {label}: REJECTED → {err['msg']}")


### What just happened?

- **`@field_validator` can transform values** — `email_must_be_lowercase` normalizes the input before returning it. The returned value replaces the raw input; `model_dump()` will show the normalized form.
- **`mode="after"`** means the validator receives the already-type-coerced value (e.g., a Python `str`, not the raw JSON value). Use `mode="before"` to intercept before type coercion.
- **`@classmethod`** is required in Pydantic v2 — unlike v1 where it was implicit.
- **Raise `ValueError`** (not `ValidationError`) inside validators — Pydantic wraps it into a proper `ValidationError` with the field location attached.


## Step 5 · `model_config` and Strict Mode

Pydantic v2 uses `model_config = ConfigDict(...)` for per-model configuration (replacing the v1 inner `class Config`). The most important settings:

| Config key | Effect |
|---|---|
| `strict=True` | Disable type coercion — `"1"` will NOT be accepted where `int` is expected |
| `from_attributes=True` | Allow building models from ORM objects (e.g., SQLAlchemy rows) |
| `populate_by_name=True` | Accept both field name and alias in input |
| `str_strip_whitespace=True` | Auto-strip leading/trailing whitespace from all `str` fields |
| `frozen=True` | Make the model immutable after construction |
| `extra="forbid"` | Raise error if unknown fields are passed (prevents silent data loss) |
| `extra="ignore"` | Silently drop unknown fields (default) |


In [ ]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError

# Default (lax) model — coerces types
class LaxModel(BaseModel):
    count: int
    name: str

# Strict model — no coercion; exact types required
class StrictModel(BaseModel):
    model_config = ConfigDict(strict=True)
    count: int
    name: str

# Lax accepts string "42" for int
lax = LaxModel(count="42", name=123)   # str → int and int → str coerced
print("Lax model — count type:", type(lax.count).__name__, "value:", lax.count)
print("Lax model — name type: ", type(lax.name).__name__, "value:", lax.name)

# Strict rejects the same input
try:
    StrictModel(count="42", name="Alice")
except ValidationError as e:
    print("\nStrict model rejects str for int field:")
    print(" ", e.errors()[0]["msg"])

# --- extra="forbid" demo ---
class CleanModel(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)
    name: str
    value: int

# str_strip_whitespace removes surrounding spaces automatically
clean = CleanModel(name="  Alice  ", value=7)
print("\nstr_strip_whitespace → name repr:", repr(clean.name))

# extra="forbid" rejects unknown fields
try:
    CleanModel(name="Bob", value=3, unknown_field="surprise")
except ValidationError as e:
    print("extra=forbid rejects unknown fields:", e.errors()[0]["msg"])


### What just happened?

- **`strict=True`** is a safety net for APIs that care about type precision — it prevents silent coercion bugs like `"false"` being treated as `True`.
- **`str_strip_whitespace=True`** is a common quality-of-life setting for user-facing models — it normalizes input before validation runs, so `"  Alice  "` is stored as `"Alice"`.
- **`extra="forbid"`** is a security pattern — it prevents clients from sending undocumented fields that could bypass application logic or confuse downstream systems.
- **`ConfigDict` is per-model** — you can mix lax and strict models in the same application, using strict for security-sensitive inputs and lax for more flexible data pipelines.


## Step 6 · Pydantic Models in FastAPI Routes — Full Integration

Combining everything: a FastAPI app that uses the full `Order` model (nested, validated, strict) as a request body and returns a validated response.


In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, ConfigDict, Field, field_validator, ValidationError
from typing import Optional
import json

# --- Models ---

class CustomerIn(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True, extra="forbid")
    name: str = Field(..., min_length=1, max_length=100)
    email: str = Field(..., pattern=r"^[\w.+-]+@[\w-]+\.[\w.]+$")

class LineItemIn(BaseModel):
    model_config = ConfigDict(extra="forbid")
    sku: str = Field(..., min_length=3, max_length=20)
    quantity: int = Field(..., ge=1, le=1000)
    unit_price: float = Field(..., gt=0)

class OrderIn(BaseModel):
    model_config = ConfigDict(extra="forbid")
    customer: CustomerIn
    line_items: list[LineItemIn] = Field(..., min_length=1, max_length=50)
    notes: Optional[str] = Field(default=None, max_length=500)

# Response model — what we return; includes computed fields
class OrderOut(BaseModel):
    order_id: str
    customer: CustomerIn
    line_items: list[LineItemIn]
    notes: Optional[str]
    total: float
    item_count: int

# --- App ---

store_app = FastAPI(title="Order API", version="1.0.0")
ORDERS: dict[str, dict] = {}  # in-memory store
order_counter = {"n": 1}

@store_app.post("/orders", response_model=OrderOut, status_code=201)
def create_order(order: OrderIn):
    """Create an order with full Pydantic validation."""
    order_id = f"ORD-{order_counter['n']:04d}"
    order_counter["n"] += 1

    # Compute total server-side — never trust client-sent totals
    total = round(
        sum(item.quantity * item.unit_price for item in order.line_items), 2
    )

    stored = {
        "order_id": order_id,
        **order.model_dump(),
        "total": total,
        "item_count": len(order.line_items),
    }
    ORDERS[order_id] = stored
    return stored

@store_app.get("/orders/{order_id}", response_model=OrderOut)
def get_order(order_id: str):
    """Retrieve a single order by ID."""
    if order_id not in ORDERS:
        raise HTTPException(status_code=404, detail=f"Order {order_id} not found")
    return ORDERS[order_id]

sc = TestClient(store_app)

# Create a valid order
payload = {
    "customer": {"name": "  Bob Smith  ", "email": "bob@example.com"},
    "line_items": [
        {"sku": "SKU-001", "quantity": 3, "unit_price": 15.00},
        {"sku": "SKU-002", "quantity": 1, "unit_price": 49.99},
    ],
    "notes": "Please gift-wrap",
}
r = sc.post("/orders", json=payload)
created = r.json()
print("POST /orders →", r.status_code)
print(json.dumps(created, indent=2))

# Retrieve the order
r = sc.get(f"/orders/{created['order_id']}")
print("\nGET /orders/ORD-0001 →", r.status_code, r.json()["total"])

# Trigger validation error: empty line_items
r = sc.post("/orders", json={"customer": {"name": "X", "email": "x@x.com"}, "line_items": []})
print("\nEmpty line_items → status:", r.status_code, "detail:", r.json()["detail"][0]["msg"])


### What just happened?

- **`response_model=OrderOut`** tells FastAPI to serialize the return value through the `OrderOut` model — any fields not in `OrderOut` are stripped (a security feature: no accidental data leakage).
- **`HTTPException(status_code=404, ...)`** is the FastAPI-idiomatic way to return error responses with the correct HTTP status code.
- **`str_strip_whitespace=True`** on `CustomerIn` means `"  Bob Smith  "` → `"Bob Smith"` before the field-level `min_length` check runs.
- **Server-side total computation** is a critical pattern: never accept `total` from the client. Compute it from line items and prices you control.


## Step 7 · `model_dump()` and `model_validate()` — Conversion Utilities

Pydantic v2 provides clean methods for converting between models, dicts, and JSON:

| Method | Input → Output | Notes |
|---|---|---|
| `Model(**dict)` | dict → model | Basic construction |
| `Model.model_validate(dict)` | dict → model | With input-mode control |
| `model.model_dump()` | model → dict | Recursive; respects `exclude`, `include` |
| `model.model_dump_json()` | model → JSON str | Faster than `json.dumps(model.model_dump())` |
| `Model.model_validate_json(str)` | JSON str → model | Parse JSON and validate in one step |


In [ ]:
from pydantic import BaseModel, Field
from typing import Optional
import json

class Config(BaseModel):
    host: str = "localhost"
    port: int = 8080
    debug: bool = False
    api_key: Optional[str] = None  # sensitive field we may want to exclude

cfg = Config(host="prod.example.com", port=443, debug=False, api_key="secret-key-abc")

# Basic dump
print("model_dump():")
print(cfg.model_dump())

# Exclude sensitive fields
print("\nmodel_dump(exclude={'api_key'}):")
print(cfg.model_dump(exclude={"api_key"}))

# Include only specific fields
print("\nmodel_dump(include={'host', 'port'}):")
print(cfg.model_dump(include={"host", "port"}))

# JSON round-trip
json_str = cfg.model_dump_json(exclude={"api_key"})
print("\nmodel_dump_json (no api_key):", json_str)

# Parse JSON back into a model
cfg2 = Config.model_validate_json(json_str)
print("model_validate_json → host:", cfg2.host, "port:", cfg2.port)

# Copy with overrides (model_copy is Pydantic v2 for .copy(update={}))
cfg_dev = cfg.model_copy(update={"host": "localhost", "debug": True})
print("\nmodel_copy(update=...) → host:", cfg_dev.host, "debug:", cfg_dev.debug)


### What just happened?

- **`model_dump(exclude=...)` / `include=...`** gives fine-grained control over serialization — useful for removing sensitive fields from API responses or logs.
- **`model_dump_json()`** uses Pydantic's Rust-backed serializer — significantly faster than `json.dumps(model.model_dump())` for large models.
- **`model_validate_json()`** is the inverse — parses a JSON string and validates it in a single call. Use this when reading stored state or webhook payloads.
- **`model_copy(update=...)`** creates a new model instance with specified fields overridden — essential for immutable update patterns (especially when `frozen=True` is set).


In [ ]:
# Challenge: Design a CourseEnrollment model for a learning platform
#
# Requirements:
#   1. model_config: extra="forbid", str_strip_whitespace=True
#   2. Fields:
#      - student_email: str (validate email format with @field_validator)
#      - course_id: str (pattern: "COURSE-" followed by 3-6 digits)
#      - seats: int (1 to 50, inclusive)
#      - discount_pct: float (0.0 to 100.0, inclusive; default 0.0)
#      - notes: Optional[str] (max 200 chars; default None)
#   3. Add a @field_validator for student_email that:
#      - Normalizes to lowercase
#      - Rejects @test.com and @example.com domains
#
# Test cases to verify:
#   valid:   email="Student@MYUNI.EDU", course_id="COURSE-101", seats=5
#   invalid: email="student@test.com"  → rejected (test domain)
#   invalid: discount_pct=150          → rejected (> 100)
#   invalid: seats=0                   → rejected (< 1)

# Your solution here:

# class CourseEnrollment(BaseModel):
#     model_config = ...
#     student_email: str
#     ...
#
#     @field_validator("student_email", mode="after")
#     @classmethod
#     def validate_email(cls, v: str) -> str:
#         ...

# Test:
# e = CourseEnrollment(student_email="Student@MYUNI.EDU", course_id="COURSE-101", seats=5)
# print(e.model_dump())


---
## Day 3 key concepts recap

| Concept | What to remember |
|---|---|
| `BaseModel` | Base class for all Pydantic models; fields without defaults are required |
| `Field(...)` | Add constraints (`ge`, `le`, `min_length`, `pattern`) and metadata to any field |
| `@field_validator` | Custom validation logic; `mode="after"` for parsed type, `mode="before"` for raw input |
| Nested models | Compose models as field types; Pydantic validates the whole tree recursively |
| `model_dump()` | Model → dict; supports `exclude`, `include`, `exclude_none`, `exclude_defaults` |
| `model_validate()` | dict → model (with input-mode control); replaces v1's `Model(**dict)` |
| `ConfigDict` | Per-model config: `strict`, `extra`, `str_strip_whitespace`, `frozen`, `from_attributes` |
| `strict=True` | Disables type coercion; catches silent bugs; use for security-sensitive models |
| `extra="forbid"` | Rejects unknown fields; prevents silent data loss and injection attacks |

> **Tip:** Pydantic v2 is 5–50× faster than v1 (Rust core). Use `model_config = ConfigDict(strict=True)` to prevent silent type coercion.

---
## What's next
**Day 4** → Response models, HTTP status codes, `HTTPException`, and custom error handlers — building production-grade API responses.

Mark Day 3 complete in your [tracker](../index.html).
